In [2]:
from pathlib import Path
import pandas as pd

# 1. 입력·출력 폴더
DATA_DIR = Path(
    "/Users/apple/Documents/team4-final-project/data/interim"
)
OUTPUT_DIR = DATA_DIR.parent / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 실제 파일명에 맞춰 수정
file_names = {
    "user": "accounts_user_preprocessed.csv",
    "friend": "accounts_friendrequest_preprocessed.csv",
    "payment": "accounts_paymenthistory_preprocessed.csv",
}

# 한국 시간으로 전처리된 created_at 기준
start_date = "2023-05-13 00:00:00"
end_date = "2024-05-08 00:00:00"  # 5월 7일 전체 포함


# 2. CSV 불러오기
# ID는 문자형으로 읽기
df_accounts_user = pd.read_csv(
    DATA_DIR / file_names["user"],
    encoding="utf-8-sig",
    dtype={"id": "string"},
)

df_accounts_friendrequest = pd.read_csv(
    DATA_DIR / file_names["friend"],
    encoding="utf-8-sig",
    dtype={
        "id": "string",
        "send_user_id": "string",
        "receive_user_id": "string",
    },
)

df_accounts_paymenthistory = pd.read_csv(
    DATA_DIR / file_names["payment"],
    encoding="utf-8-sig",
    dtype={
        "id": "string",
        "user_id": "string",
    },
)


# 3. created_at을 datetime으로 변환
for df in [
    df_accounts_user,
    df_accounts_friendrequest,
    df_accounts_paymenthistory,
]:
    df["created_at"] = pd.to_datetime(
        df["created_at"],
        format="mixed",
        errors="raise",
    )


# 4. accounts_user 필터링
# 시작일 제한 없이 2024-05-07까지 포함
df_accounts_user_filtered = df_accounts_user.loc[
    df_accounts_user["created_at"] < end_date
].copy()


# 5. accounts_friendrequest 필터링
# 지정 기간에 생성된 요청 중 수락 상태(A)
df_accounts_friendrequest_filtered = df_accounts_friendrequest.loc[
    (df_accounts_friendrequest["status"] == "A")
    & (df_accounts_friendrequest["created_at"] >= start_date)
    & (df_accounts_friendrequest["created_at"] < end_date)
].copy()


# 6. accounts_paymenthistory 필터링
# 지정 기간의 결제 기록
df_accounts_paymenthistory_filtered = df_accounts_paymenthistory.loc[
    (df_accounts_paymenthistory["created_at"] >= start_date)
    & (df_accounts_paymenthistory["created_at"] < end_date)
].copy()


# 7. CSV 저장 및 전후 행 수 확인
results = [
    (
        "accounts_user",
        df_accounts_user,
        df_accounts_user_filtered,
    ),
    (
        "accounts_friendrequest",
        df_accounts_friendrequest,
        df_accounts_friendrequest_filtered,
    ),
    (
        "accounts_paymenthistory",
        df_accounts_paymenthistory,
        df_accounts_paymenthistory_filtered,
    ),
]

for table_name, original_df, filtered_df in results:
    output_path = (
        OUTPUT_DIR / f"{table_name}_preprocessed_filtered.csv"
    )

    filtered_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"{table_name}: "
        f"{len(original_df):,} → {len(filtered_df):,}행"
    )
    print(f"저장 파일: {output_path.name}\n")

print(f"저장 폴더: {OUTPUT_DIR}")

accounts_user: 677,080 → 677,075행
저장 파일: accounts_user_preprocessed_filtered.csv

accounts_friendrequest: 17,147,175 → 7,793,140행
저장 파일: accounts_friendrequest_preprocessed_filtered.csv

accounts_paymenthistory: 95,140 → 95,137행
저장 파일: accounts_paymenthistory_preprocessed_filtered.csv

저장 폴더: /Users/apple/Documents/team4-final-project/data/processed
